In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor


DATA_PATH = "최종_용인_실거래가_통합_결측채움.csv"
MODEL_PATH = "xgb_price_unit_ensemble_model.joblib"
METRICS_PATH = "xgb_price_unit_ensemble_metrics.json"

RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_CATEGORY_COUNT = 50
SMOOTHING = 20

COL_SIGUNGU = "시군구"
COL_AREA = "연면적(㎡)"
COL_LAND = "대지면적(㎡)"
COL_TYPE = "주택유형"
COL_YEAR = "건축년도"
COL_PRICE = "거래금액(만원)"
COL_TRADE_YM = "계약년월"
COL_TRADE_D = "계약일"
COL_ROAD_COND = "도로조건"


def normalize_money(value):
    text = str(value).replace(",", "").strip()
    return pd.to_numeric(text, errors="coerce")


def clean_category(value):
    text = str(value).strip()
    if text in {"", "-", "nan", "None"}:
        return "기타"
    return text


def extract_gu(addr):
    text = str(addr)
    for gu in ["수지구", "기흥구", "처인구"]:
        if gu in text:
            return gu
    return "기타"


def extract_eup_myeon_dong(addr):
    parts = str(addr).split()
    if not parts:
        return "기타"
    return parts[-1]


def prepare_dataframe(df):
    out = df.copy()

    out["contract_ym"] = pd.to_numeric(out[COL_TRADE_YM], errors="coerce")
    out["trade_year"] = (out["contract_ym"] // 100).astype("Int64")
    out["trade_month"] = (out["contract_ym"] % 100).astype("Int64")
    out["trade_day"] = pd.to_numeric(out[COL_TRADE_D], errors="coerce")

    out["price"] = out[COL_PRICE].apply(normalize_money)
    out["total_area"] = pd.to_numeric(out[COL_AREA], errors="coerce")
    out["land_area"] = pd.to_numeric(out[COL_LAND], errors="coerce")
    out["build_year"] = pd.to_numeric(out[COL_YEAR], errors="coerce")

    out = out.dropna(
        subset=[
            "price",
            "total_area",
            "land_area",
            "build_year",
            "trade_year",
            "trade_month",
            "trade_day",
        ]
    )

    out = out[
        (out["price"] > 0)
        & (out["total_area"] > 0)
        & (out["land_area"] > 0)
    ]

    out["trade_year"] = out["trade_year"].astype(int)
    out["trade_month"] = out["trade_month"].astype(int)
    out["trade_day"] = out["trade_day"].astype(int)
    out["build_year"] = out["build_year"].astype(int)

    out["building_age"] = out["trade_year"] - out["build_year"]
    out = out[(out["building_age"] >= 0) & (out["building_age"] <= 150)]

    out["far"] = out["total_area"] / out["land_area"]
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=["far"])
    out = out[(out["far"] > 0) & (out["far"] < 20)]

    out["log_total_area"] = np.log1p(out["total_area"])
    out["log_land_area"] = np.log1p(out["land_area"])
    out["land_to_total_ratio"] = out["land_area"] / out["total_area"]
    out["total_to_land_ratio"] = out["total_area"] / out["land_area"]

    out["house_type"] = out[COL_TYPE].map(clean_category)
    out["gu"] = out[COL_SIGUNGU].apply(extract_gu)
    out["dong"] = out[COL_SIGUNGU].apply(extract_eup_myeon_dong).map(clean_category)
    out["road_condition"] = out[COL_ROAD_COND].map(clean_category)

    out["gu_house_type"] = out["gu"] + "_" + out["house_type"]
    out["dong_house_type"] = out["dong"] + "_" + out["house_type"]
    out["dong_road"] = out["dong"] + "_" + out["road_condition"]

    out["contract_date_order"] = (
        out["trade_year"] * 10000
        + out["trade_month"] * 100
        + out["trade_day"]
    )

    price_lower = out["price"].quantile(0.01)
    price_upper = out["price"].quantile(0.99)

    out = out[(out["price"] >= price_lower) & (out["price"] <= price_upper)]
    out = out.sort_values("contract_date_order").reset_index(drop=True)

    out["price_per_total_area"] = out["price"] / out["total_area"]

    return out


def time_based_split(df, test_size=0.2):
    split_idx = int(len(df) * (1 - test_size))
    split_idx = min(max(split_idx, 1), len(df) - 1)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()


class MultiTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, smoothing=20):
        self.columns = columns
        self.smoothing = smoothing

    def fit(self, X, y, sample_weight=None):
        self.global_mean_ = float(np.mean(y))
        self.mappings_ = {}

        X_temp = X.copy()
        X_temp["_target"] = np.asarray(y)

        for col in self.columns:
            stats = X_temp.groupby(col)["_target"].agg(["mean", "count"])

            mapping = (
                (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing)
                / (stats["count"] + self.smoothing)
            )

            self.mappings_[col] = mapping

        return self

    def transform(self, X):
        X_out = X.copy()

        for col in self.columns:
            X_out[f"{col}_target_avg"] = (
                X_out[col]
                .map(self.mappings_[col])
                .fillna(self.global_mean_)
            )

        return X_out


TARGET_ENCODE_COLS = [
    "gu",
    "dong",
    "house_type",
    "gu_house_type",
    "dong_house_type",
    "dong_road",
]

NUM_FEATURES = [
    "total_area",
    "land_area",
    "log_total_area",
    "log_land_area",
    "far",
    "land_to_total_ratio",
    "total_to_land_ratio",
    "building_age",
    "build_year",
    "trade_year",
    "trade_month",
    "trade_day",
    "gu_target_avg",
    "dong_target_avg",
    "house_type_target_avg",
    "gu_house_type_target_avg",
    "dong_house_type_target_avg",
    "dong_road_target_avg",
]

CAT_FEATURES = [
    "gu",
    "dong",
    "road_condition",
    "house_type",
    "gu_house_type",
    "dong_house_type",
    "dong_road",
]

BASE_FEATURES = [
    "total_area",
    "land_area",
    "log_total_area",
    "log_land_area",
    "far",
    "land_to_total_ratio",
    "total_to_land_ratio",
    "building_age",
    "build_year",
    "trade_year",
    "trade_month",
    "trade_day",
] + CAT_FEATURES


def build_pipeline(params):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), NUM_FEATURES),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                min_frequency=MIN_CATEGORY_COUNT,
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                CAT_FEATURES,
            ),
        ]
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        **params,
    )

    return Pipeline(
        steps=[
            (
                "target_encoder",
                MultiTargetEncoder(
                    columns=TARGET_ENCODE_COLS,
                    smoothing=SMOOTHING,
                ),
            ),
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


def evaluate(y_true, pred):
    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)

    y_true_array = np.asarray(y_true, dtype=float)
    pred_array = np.asarray(pred, dtype=float)

    nonzero_mask = y_true_array != 0

    mape = (
        np.mean(
            np.abs(
                (y_true_array[nonzero_mask] - pred_array[nonzero_mask])
                / y_true_array[nonzero_mask]
            )
        )
        * 100
        if nonzero_mask.any()
        else np.nan
    )

    wmape = (
        np.sum(np.abs(y_true_array - pred_array))
        / np.sum(np.abs(y_true_array))
        * 100
        if np.sum(np.abs(y_true_array)) != 0
        else np.nan
    )

    return {
        "MAE_만원": float(mae),
        "RMSE_만원": float(rmse),
        "R2": float(r2),
        "MAPE_%": float(mape),
        "WMAPE_%": float(wmape),
    }


def analyze_jeonse_risk(
    estimated_price_manwon,
    mortgage_amount_manwon,
    deposit_manwon,
    liquidation_rate=0.85,
):
    eps = 1e-9

    mortgage_ltv = mortgage_amount_manwon / estimated_price_manwon
    total_ltv = (mortgage_amount_manwon + deposit_manwon) / estimated_price_manwon

    expected_liquidation_value = estimated_price_manwon * liquidation_rate
    recoverable_deposit = max(0, expected_liquidation_value - mortgage_amount_manwon)
    recoverable_deposit = min(deposit_manwon, recoverable_deposit)

    expected_loss = max(0, deposit_manwon - recoverable_deposit)

    recovery_rate = (
        recoverable_deposit / deposit_manwon
        if deposit_manwon > 0
        else 1.0
    )

    if (
        mortgage_ltv <= 0.50 + eps
        and total_ltv <= 0.70 + eps
        and recovery_rate >= 0.95 - eps
    ):
        grade = "안전"
    elif (
        mortgage_ltv <= 0.70 + eps
        and total_ltv <= 0.90 + eps
        and recovery_rate >= 0.80 - eps
    ):
        grade = "주의"
    else:
        grade = "위험"

    return {
        "estimated_price_manwon": float(estimated_price_manwon),
        "mortgage_amount_manwon": float(mortgage_amount_manwon),
        "deposit_manwon": float(deposit_manwon),
        "mortgage_ltv": float(mortgage_ltv),
        "total_ltv": float(total_ltv),
        "liquidation_rate": float(liquidation_rate),
        "recoverable_deposit_manwon": float(recoverable_deposit),
        "expected_loss_manwon": float(expected_loss),
        "deposit_recovery_rate": float(recovery_rate),
        "safety_grade": grade,
    }


def main():
    df = pd.read_csv(DATA_PATH)
    data = prepare_dataframe(df)

    train_df, test_df = time_based_split(data, test_size=TEST_SIZE)

    X_train = train_df[BASE_FEATURES]
    X_test = test_df[BASE_FEATURES]

    y_train_raw = train_df["price"]
    y_train_log = np.log1p(train_df["price"])
    y_train_unit = train_df["price_per_total_area"]

    y_test = test_df["price"]

    weight_base_year = train_df["trade_year"].min()

    sample_weight = 0.7 + 0.3 * (
        (train_df["trade_year"] - weight_base_year)
        / max(1, train_df["trade_year"].max() - weight_base_year)
    )

    raw_params = {
        "n_estimators": 600,
        "max_depth": 5,
        "learning_rate": 0.025,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 2.0,
    }

    log_params = {
        "n_estimators": 600,
        "max_depth": 4,
        "learning_rate": 0.025,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
    }

    unit_params = {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 2.0,
    }

    cv = TimeSeriesSplit(n_splits=5)

    weight_candidates = [
        (0.50, 0.50, 0.00),
        (0.45, 0.45, 0.10),
        (0.40, 0.40, 0.20),
        (0.50, 0.35, 0.15),
        (0.60, 0.30, 0.10),
        (0.35, 0.50, 0.15),
    ]

    weight_scores = {w: [] for w in weight_candidates}

    for train_idx, valid_idx in cv.split(X_train):
        X_cv_train = X_train.iloc[train_idx]
        X_cv_valid = X_train.iloc[valid_idx]

        y_cv_train_raw = y_train_raw.iloc[train_idx]
        y_cv_train_log = y_train_log.iloc[train_idx]
        y_cv_train_unit = y_train_unit.iloc[train_idx]
        y_cv_valid_raw = y_train_raw.iloc[valid_idx]

        w_cv_train = sample_weight.iloc[train_idx]

        raw_pipeline = build_pipeline(raw_params)
        log_pipeline = build_pipeline(log_params)
        unit_pipeline = build_pipeline(unit_params)

        raw_pipeline.fit(
            X_cv_train,
            y_cv_train_raw,
            target_encoder__sample_weight=w_cv_train,
            model__sample_weight=w_cv_train,
        )

        log_pipeline.fit(
            X_cv_train,
            y_cv_train_log,
            target_encoder__sample_weight=w_cv_train,
            model__sample_weight=w_cv_train,
        )

        unit_pipeline.fit(
            X_cv_train,
            y_cv_train_unit,
            target_encoder__sample_weight=w_cv_train,
            model__sample_weight=w_cv_train,
        )

        raw_pred = np.maximum(raw_pipeline.predict(X_cv_valid), 0)

        log_pred = log_pipeline.predict(X_cv_valid)
        log_pred = np.maximum(np.expm1(log_pred), 0)

        unit_pred = unit_pipeline.predict(X_cv_valid)
        unit_price_pred = np.maximum(unit_pred * X_cv_valid["total_area"].values, 0)

        for weights in weight_candidates:
            a, b, c = weights
            ensemble_pred = a * raw_pred + b * log_pred + c * unit_price_pred
            mae = mean_absolute_error(y_cv_valid_raw, ensemble_pred)
            weight_scores[weights].append(mae)

    weight_result = {
        str(weights): float(np.mean(scores))
        for weights, scores in weight_scores.items()
    }

    best_weights = min(weight_scores, key=lambda w: np.mean(weight_scores[w]))

    print("\n===== CV Ensemble Weight Search =====")
    for weights, scores in weight_scores.items():
        print(f"weights={weights} | CV MAE={np.mean(scores):.4f}")

    print(f"best_weights: {best_weights}")

    final_raw_pipeline = build_pipeline(raw_params)
    final_log_pipeline = build_pipeline(log_params)
    final_unit_pipeline = build_pipeline(unit_params)

    final_raw_pipeline.fit(
        X_train,
        y_train_raw,
        target_encoder__sample_weight=sample_weight,
        model__sample_weight=sample_weight,
    )

    final_log_pipeline.fit(
        X_train,
        y_train_log,
        target_encoder__sample_weight=sample_weight,
        model__sample_weight=sample_weight,
    )

    final_unit_pipeline.fit(
        X_train,
        y_train_unit,
        target_encoder__sample_weight=sample_weight,
        model__sample_weight=sample_weight,
    )

    raw_test_pred = np.maximum(final_raw_pipeline.predict(X_test), 0)

    log_test_pred = final_log_pipeline.predict(X_test)
    log_test_pred = np.maximum(np.expm1(log_test_pred), 0)

    unit_test_pred = final_unit_pipeline.predict(X_test)
    unit_test_price_pred = np.maximum(unit_test_pred * X_test["total_area"].values, 0)

    a, b, c = best_weights
    pred = a * raw_test_pred + b * log_test_pred + c * unit_test_price_pred
    pred = np.maximum(pred, 0)

    metrics = evaluate(y_test, pred)

    sample_estimated_price = float(pred[0])

    sample_risk = analyze_jeonse_risk(
        estimated_price_manwon=sample_estimated_price,
        mortgage_amount_manwon=sample_estimated_price * 0.45,
        deposit_manwon=sample_estimated_price * 0.25,
    )

    print("\n===== Final Test Metrics =====")
    print(f"rows_total: {len(data):,}")
    print(f"train_rows: {len(train_df):,}")
    print(f"test_rows: {len(test_df):,}")
    print(f"test_period: {test_df['contract_ym'].min()} ~ {test_df['contract_ym'].max()}")
    print(f"raw_params: {raw_params}")
    print(f"log_params: {log_params}")
    print(f"unit_params: {unit_params}")
    print(f"best_weights: {best_weights}")

    for k, v in metrics.items():
        print(f"{k}: {v:,.4f}")

    print("\n===== Jeonse Risk Example =====")
    print(f"estimated_price: {sample_risk['estimated_price_manwon']:,.0f} 만원")
    print(f"mortgage_amount: {sample_risk['mortgage_amount_manwon']:,.0f} 만원")
    print(f"deposit: {sample_risk['deposit_manwon']:,.0f} 만원")
    print(f"mortgage_ltv: {sample_risk['mortgage_ltv']:.2%}")
    print(f"total_ltv: {sample_risk['total_ltv']:.2%}")
    print(f"recoverable_deposit: {sample_risk['recoverable_deposit_manwon']:,.0f} 만원")
    print(f"expected_loss: {sample_risk['expected_loss_manwon']:,.0f} 만원")
    print(f"safety_grade: {sample_risk['safety_grade']}")

    result = {
        "rows_total": int(len(data)),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "test_period": [
            int(test_df["contract_ym"].min()),
            int(test_df["contract_ym"].max()),
        ],
        "model_type": "raw_log_unit_price_ensemble",
        "smoothing": SMOOTHING,
        "raw_params": raw_params,
        "log_params": log_params,
        "unit_params": unit_params,
        "weight_result": weight_result,
        "best_weights": list(best_weights),
        "metrics": metrics,
        "risk_example": sample_risk,
        "target_encode_cols": TARGET_ENCODE_COLS,
        "num_features": NUM_FEATURES,
        "cat_features": CAT_FEATURES,
        "base_features": BASE_FEATURES,
    }

    joblib.dump(
        {
            "raw_pipeline": final_raw_pipeline,
            "log_pipeline": final_log_pipeline,
            "unit_pipeline": final_unit_pipeline,
            "best_weights": best_weights,
            "smoothing": SMOOTHING,
            "raw_params": raw_params,
            "log_params": log_params,
            "unit_params": unit_params,
            "target_encode_cols": TARGET_ENCODE_COLS,
            "num_features": NUM_FEATURES,
            "cat_features": CAT_FEATURES,
            "base_features": BASE_FEATURES,
            "metrics": result,
        },
        MODEL_PATH,
    )

    Path(METRICS_PATH).write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"\nmodel saved: {MODEL_PATH}")
    print(f"metrics saved: {METRICS_PATH}")


if __name__ == "__main__":
    main()